# Can a prior mean be estimated from the seven patients?

The seven patient fiber fields, on the mesh of patient 01: patient 01 itself and the six others mapped to it (`patient_data/mapped_to_patient_01/`, UAC mapping with nearest-neighbor transfer), each raw and smoothed at scale 3. If the fields shared a common mean pattern, the mean of six patients would predict the seventh, and the fields would agree more than random orientations do. The checks below test that, in the frame the inference uses: the fiber angle against the basis vectors of patient 01. Angles are axial, so all differences are taken modulo $\pi$ and all means are axial means (of $e^{2i\theta}$).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PATIENT_DATA = Path.cwd().parent / "patient_data"
MAPPED = PATIENT_DATA / "mapped_to_patient_01"
SMOOTHED = "smoothed_scale_3"
PATIENTS = [f"patient_0{i}" for i in range(1, 8)]
RANDOM_ERROR_DEG = 45.0  # mean axial error between two independent random orientations

basis_vectors = np.load(PATIENT_DATA / "patient_01" / "basis_vecs.npy")


def angles_of(fiber_field):
    """Fiber angle against the basis vectors of patient 01."""
    return np.arctan2(
        np.einsum("ij,ij->i", fiber_field, basis_vectors[..., 1]),
        np.einsum("ij,ij->i", fiber_field, basis_vectors[..., 0]),
    )


def load_angles(smoothed):
    folder = SMOOTHED if smoothed else ""
    fields = [np.load(PATIENT_DATA / "patient_01" / folder / "fiber_field.npy")]
    fields += [np.load(MAPPED / folder / f"fiber_field_from_{name}.npy") for name in PATIENTS[1:]]
    return np.stack([angles_of(field) for field in fields])  # (7, num_simplices)


def wrap(angle):
    return np.angle(np.exp(2j * angle)) / 2


def axial_mean(angles, axis=0):
    return np.angle(np.mean(np.exp(2j * angles), axis=axis)) / 2


def error_deg(a, b):
    return np.degrees(np.abs(wrap(a - b)))


ANGLES = {"raw": load_angles(False), "smoothed (scale 3)": load_angles(True)}
print({name: angles.shape for name, angles in ANGLES.items()})

## 1. How much do the patients agree?

Mean axial difference between each pair of fields, in degrees. Independent random orientations differ by 45 degrees on average, so values near 45 mean no agreement. The diagonal is zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for axis, (name, angles) in zip(axes, ANGLES.items(), strict=True):
    matrix = np.array([[error_deg(a, b).mean() for b in angles] for a in angles])
    image = axis.imshow(matrix, vmin=0, vmax=RANDOM_ERROR_DEG, cmap="viridis")
    for i, j in np.ndindex(matrix.shape):
        axis.text(j, i, f"{matrix[i, j]:.0f}", ha="center", va="center", color="white", fontsize=8)
    axis.set_xticks(range(7), [p[-2:] for p in PATIENTS])
    axis.set_yticks(range(7), [p[-2:] for p in PATIENTS])
    off_diagonal = matrix[~np.eye(7, dtype=bool)]
    axis.set_title(f"{name}: mean pairwise difference {off_diagonal.mean():.1f} deg")
fig.colorbar(image, ax=axes, label="mean axial difference [deg]")

## 2. Is the agreement at each location larger than chance?

At every simplex, the mean resultant length $R = |\frac17 \sum_k e^{2i\theta_k}|$ of the seven angles: one if all patients have the same fiber there, and about $0.33$ typically for seven independent random orientations. The histogram compares the fields with that null (simulated).

In [ ]:
rng = np.random.default_rng(0)
null_resultant = np.abs(np.mean(np.exp(2j * rng.uniform(-np.pi / 2, np.pi / 2, (7, 200_000))), axis=0))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for axis, (name, angles) in zip(axes, ANGLES.items(), strict=True):
    resultant = np.abs(np.mean(np.exp(2j * angles), axis=0))
    bins = np.linspace(0, 1, 41)
    axis.hist(null_resultant, bins, density=True, alpha=0.5, label="random orientations")
    axis.hist(resultant, bins, density=True, alpha=0.5, label="patients")
    axis.set_title(
        f"{name}: median R {np.median(resultant):.2f} (null {np.median(null_resultant):.2f}), "
        f"R > 0.8 at {np.mean(resultant > 0.8):.1%} (null {np.mean(null_resultant > 0.8):.1%})"
    )
    axis.set_xlabel("mean resultant length R over the 7 patients")
axes[0].set_ylabel("density")
axes[0].legend()
fig.tight_layout()

## 3. Leave-one-patient-out prediction

For each patient, the axial mean of the other six is the estimate of the prior mean. Its error against the left-out patient is compared with the zero field the inference uses, with a constant (the single axial mean angle of all six other fields, no spatial structure) and with an oracle constant (the left-out patient's own mean angle, which no estimate can know). A useful spatial mean estimate would be clearly below the constant and the 45 degree random level.

In [ ]:
rows = {}
for name, angles in ANGLES.items():
    for held_out in range(7):
        others = np.delete(angles, held_out, axis=0)
        target = angles[held_out]
        rows[(name, PATIENTS[held_out])] = {
            "mean of others": error_deg(axial_mean(others), target).mean(),
            "constant of others": error_deg(axial_mean(others.ravel()), target).mean(),
            "zero field": error_deg(0.0, target).mean(),
            "oracle constant": error_deg(axial_mean(target), target).mean(),
            "within 10 deg (mean of others)": np.mean(error_deg(axial_mean(others), target) < 10),
        }

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
width = 0.2
for axis, name in zip(axes, ANGLES, strict=True):
    for offset, key in enumerate(("mean of others", "constant of others", "zero field", "oracle constant")):
        values = [rows[(name, p)][key] for p in PATIENTS]
        axis.bar(np.arange(7) + (offset - 1.5) * width, values, width, label=key)
    axis.axhline(RANDOM_ERROR_DEG, color="gray", linestyle=":", label="random orientations")
    axis.set_xticks(range(7), [p[-2:] for p in PATIENTS])
    axis.set_xlabel("left-out patient")
    axis.set_title(name)
axes[0].set_ylabel("mean axial error [deg]")
axes[0].legend(fontsize=8)
fig.tight_layout()

print(f"{'fields':<20}{'mean of others':>16}{'const. of others':>18}{'zero field':>12}{'oracle const.':>15}{'< 10 deg':>10}")
for name in ANGLES:
    mean_row = {k: np.mean([rows[(name, p)][k] for p in PATIENTS]) for k in rows[(name, PATIENTS[0])]}
    print(
        f"{name:<20}{mean_row['mean of others']:>16.1f}{mean_row['constant of others']:>18.1f}"
        f"{mean_row['zero field']:>12.1f}"
        f"{mean_row['oracle constant']:>15.1f}{mean_row['within 10 deg (mean of others)']:>10.1%}"
    )

## 4. The mean field itself

The axial mean of the seven fields and the concentration $R$ behind it, on the mesh of patient 01, wrapped modulo $\pi$. The mean has structure only where $R$ is large; elsewhere it is a mean of fibers that point in different directions and is not an estimate of anything.

In [ ]:
import pyvista as pv

from bayes_cep.mesh.io import load_pyvista_mesh

pv.set_jupyter_backend("trame")
mesh = load_pyvista_mesh(PATIENT_DATA / "patient_01" / "mesh.vtu")
for name, angles in ANGLES.items():
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["mean angle [rad]"] = axial_mean(angles)
    plot_mesh.cell_data["R"] = np.abs(np.mean(np.exp(2j * angles), axis=0))
    plotter = pv.Plotter(shape=(1, 2))
    plotter.add_mesh(plot_mesh, scalars="mean angle [rad]", cmap="hsv", clim=(-np.pi / 2, np.pi / 2))
    plotter.add_text(f"{name}: axial mean of 7", font_size=10)
    plotter.subplot(0, 1)
    plotter.add_mesh(plot_mesh, scalars="R", cmap="viridis", clim=(0, 1))
    plotter.add_text("concentration R", font_size=10)
    plotter.link_views()
    plotter.show()

## Reading

- **Little shared structure.** The pairwise differences average 43 degrees (raw, range 40 to 47) and 41 (smoothed, range 34 to 48), against 45 for random orientations, and the concentration $R$ over the seven patients has a median of 0.39 (raw) and 0.44 (smoothed), against 0.32 for random orientations. Only a few percent of the simplices have $R > 0.8$.
- **The mean of six patients is no better than a constant.** Its leave-one-out error is about the same as that of a single constant angle and close to the oracle constant. The spatial pattern of the mean adds nothing a number does not give. Smoothing lowers the errors of all estimates by a few degrees, but not the gap between the mean and the constant.
- **What is shared is an overall orientation**: the global axial mean angles of the patients lie between 57 and 101 degrees in this frame, which is why any of these constants beats the zero field by 10 to 15 degrees. A constant offset costs the zero-mean prior almost nothing (about $10^{-2}$ times $N/2$ in the prior cost), so the data fix it, and it is not a reason to change the prior mean.
- Together: no spatially varying prior mean can be estimated from these seven fields, so the prior mean stays zero.